In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import (
    HistGradientBoostingClassifier, 
    RandomForestClassifier,
    ExtraTreesClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb

# 1. Reproducibility and Settings
pd.set_option('display.max_columns', 30)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 2. Data Loading
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

X = train.drop(columns=['id', 'smoking'])
y = train['smoking']
X_test = test.drop(columns=['id'])

# 3. Feature Engineering
def clean_data(df):
    df_clean = df.copy()
    
    # Handle the 9.9 sentinel value in eyesight (blind/not measurable)
    for col in ['eyesight(left)', 'eyesight(right)']:
        df_clean[col + '_missing'] = (df_clean[col] == 9.9).astype(int)
        df_clean[col] = df_clean[col].replace(9.9, np.nan)
        # Impute with median for the tree models (linear models will be handled via scaling/imputation in pipeline if needed)
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        
    return df_clean

X = clean_data(X)
X_test = clean_data(X_test)

# Define column groups for specific transformations
skewed_cols = ['triglyceride', 'Gtp', 'ALT', 'AST', 'fasting blood sugar']
standard_cols = [c for c in X.columns if c not in skewed_cols]

# Apply log1p transform to skewed columns to assist the LogisticRegression model
preprocessor = ColumnTransformer(
    transformers=[
        ('log', FunctionTransformer(np.log1p), skewed_cols),
        ('std', StandardScaler(), standard_cols)
    ],
    remainder='passthrough'
)

# 4. Define Base Models
# Added XGBoost, LightGBM, and ExtraTrees to the original HistGradientBoosting and RandomForest
models = {
    'hgb': HistGradientBoostingClassifier(
        random_state=RANDOM_STATE, 
        max_iter=500,
        learning_rate=0.05,
        max_leaf_nodes=31
    ),
    'xgb': xgb.XGBClassifier(
        random_state=RANDOM_STATE,
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='auc'
    ),
    'lgb': lgb.LGBMClassifier(
        random_state=RANDOM_STATE,
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8
    ),
    'rf': RandomForestClassifier(
        random_state=RANDOM_STATE, 
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=4,
        n_jobs=-1
    ),
    'et': ExtraTreesClassifier(
        random_state=RANDOM_STATE,
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=4,
        n_jobs=-1
    ),
    'lr': LogisticRegression(
        random_state=RANDOM_STATE, 
        max_iter=1000, 
        C=0.1
    )
}

# 5. Build the Stacking Ensemble
# The StackingClassifier automatically handles out-of-fold predictions to train the meta-model
estimators = [(name, model) for name, model in models.items()]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=RANDOM_STATE, C=1.0),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
    passthrough=False 
)

# 6. Final Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('ensemble', stacking_clf)
])

# 7. Cross-Validation Evaluation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"OOF ROC AUC Scores: {cv_scores}")
print(f"Mean ROC AUC: {cv_scores.mean():.5f} +/- {cv_scores.std():.5f}")

# 8. Train on Full Dataset and Predict
pipeline.fit(X, y)
test_preds = pipeline.predict_proba(X_test)[:, 1]

# 9. Create Submission
submission = pd.DataFrame({
    'id': test['id'],
    'smoking': test_preds
})
submission.to_csv('submission.csv', index=False)
print("Submission saved to 'submission.csv'")